# GPU cumulative AO replay
Download a completed Bouchet demo into `results/bouchet/gpu_demo/JOB_ID` first. This notebook reads saved coefficients, timestamps and object estimates; it performs no optimization or inverse object fits. See [README.md](README.md) for queue and download commands.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
while not (ROOT / 'optimization').is_dir():
    if ROOT == ROOT.parent:
        raise RuntimeError('Open this notebook inside the repository.')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import HTML, display
from systematic_testing.gpu_demo.plot_demo import load_run, plot_rms, make_plots, make_animations
candidates = list((ROOT / 'results/bouchet/gpu_demo').glob('*/run.json'))
if not candidates:
    raise FileNotFoundError('Download a demo result directory before running this notebook.')
RUN_DIR = max(candidates, key=lambda p: p.stat().st_mtime).parent
# Override RUN_DIR here to select a particular downloaded JOB_ID.
print('Reading:', RUN_DIR)
metadata, scene, history = load_run(RUN_DIR)
print('Status:', metadata['status'], '| Hardware:', metadata.get('hardware', metadata['hostname']))
print(f"Core: {metadata['core_seconds']:.3f} s; final applied RMS: {history['applied_residual_rms'][-1]:.6f} waves")
display(pd.DataFrame(metadata['loops'])[['loop', 'fit_images', 'core_seconds', 'screening_seconds', 'accepted_steps', 'success', 'message']])

The curves separate phase-estimation error from the residual after each clipped SLM correction. Core time includes workspace setup, screening and Newton steps; it excludes simulated acquisitions, diagnostics and replay.

In [ ]:
fig, ax = plot_rms(metadata, scene, history, by_time=True)
plt.show()
fig, ax = plot_rms(metadata, scene, history, by_time=False)
plt.show()
print('Static plots:', make_plots(RUN_DIR))

Playback uses the original testing notebooks' polar-phase and 3D-object helper, supplied with the archived GPU objects. Phase frames show the estimate of the original static aberration (fitted residual minus applied SLM). `STRIDE` changes playback only; the final state of each loop is always retained. Frames have equal display duration, with measured core timestamps in their titles.

In [ ]:
STRIDE = 1
FPS = 4
plt.rcParams['animation.embed_limit'] = 100
aberration_anim, object_anim = make_animations(RUN_DIR, stride=STRIDE, fps=FPS, percentile=92)
display(HTML(aberration_anim.to_jshtml()))
display(HTML(object_anim.to_jshtml()))

In [ ]:
FIGURES = RUN_DIR / 'figures'
FIGURES.mkdir(exist_ok=True)
aberration_anim.save(FIGURES / 'aberration.gif', writer='pillow', fps=FPS)
object_anim.save(FIGURES / 'object.gif', writer='pillow', fps=FPS)